<a href="https://colab.research.google.com/github/vishaljoshi24/DungeonsAndDragonsTurnClassification/blob/main/turn_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone 'https://github.com/vishaljoshi24/Dungeons-and-Dragons-Turn-Classification/'

In [ ]:
# !pip install dspy==3.1.0
!pip install dspy==3.2.1

In [ ]:
import pandas as pd
import dspy

# Data

In [ ]:
training_df = pd.read_excel('further_testing_articulation_codes_set.xlsx')

In [ ]:
training_df

In [ ]:
training_df.drop(columns=['Jack\'s Codes'], inplace=True)

In [ ]:
training_df.drop(columns=['Agreed Codes'], inplace=True)

In [ ]:
training_df.drop(columns=['Notes'], inplace=True)

In [ ]:
training_df.drop(columns=['Column1'], inplace=True)

In [ ]:
training_df

In [ ]:
trainset = []

for context, current_turn, category in training_df.values:
    examples = dspy.Example(context=context, turn=current_turn, category=category).with_inputs("context", "turn")
    trainset.append(examples)

In [ ]:
trainset

# Language Model

In [ ]:
lm = dspy.LM('ollama_chat/qwen3:8b', api_base = 'http://localhost:11434', api_key='', max_tokens=4096)
dspy.configure(lm=lm)

# Signatures

In [ ]:
from typing import Literal

class TurnClassifier(dspy.Signature):
    """Given the context for a Dungeons & Dragons game turn and the game turn itself, classify the turn."""
    context: str = dspy.InputField(desc = "The three previous game turns which describe a player's action or their dialogue.")
    question: str = dspy.InputField (desc="The current turn taken by a player, which can include a description of an action or a piece of dialogue.")
    response: Literal['knowledge request',
                      'knowledge update',
                      'knowledge share',
                      'knowledge confirmation',
                      'argumentation',
                      'resource use',
                      'resource share',
                      'resource aid',
                      'resource request',
                      'enact narration',
                      'deterministic action'
                      ] = dspy.OutputField()

class PlayerInstruction(dspy.Signature):
  category: Literal['knowledge request',
                      'knowledge update',
                      'knowledge share',
                      'knowledge confirmation',
                      'argumentation',
                      'resource use',
                      'resource share',
                      'resource aid',
                      'resource request',
                      'enact narration',
                      'deterministic action'
                    ] = dspy.InputField()
  player_instruction:str = dspy.OutputField(desc="instruction on how to behave within a D&D game.")


# Modules

In [ ]:
class ClassifyTurns(dspy.Module):
  def __init__(self):
    self.classifier = dspy.ChainOfThought(TurnClassifier, caching=False)

  def forward(self, context, question, **kwargs):
    prediction = self.classifier(context=context, question=question)
    return prediction

In [ ]:
classify = ClassifyTurns()
def classify_turn(context, question):
    try:
        predicted_category = classify(context=context, question=question)
        return predicted_category
    except Exception as e:
        return 0

In [ ]:
# class PromptGenerator(dspy.Module):
#   def __init__(self):
#     self.classifier = classify
#     self.generator = dspy.ChainOfThought(PlayerInstruction, caching=False)

#   def forward(self, context, question, **kwargs):
#     pred_category = classify(context=context, question=question)
#     prompt = self.generator(category=pred_category)
#     return prompt, pred_category

# prompt_generator = PromptGenerator()


In [ ]:
# prompt_generator(trainset[1]['context'], trainset[1]['question'])

In [ ]:
predictions = []

for i in range(len(trainset[0:10])):
    predictions.append(zeroR_classify_turn(trainset[i]['context'], trainset[i]['turn'], 'enact narration'))

In [ ]:
predictions

In [ ]:
predicted_categories = []

for i in range(len(predictions)):
  predicted_categories.append(predictions[i].zeroRclass)

In [ ]:
predicted_categories

In [ ]:
true_categories = []

for i in range(len(trainset[0:10])):
  true_categories.append(trainset[i]['category'])

In [ ]:
true_categories

In [ ]:
label_list = [
    'knowledge request',
    'knowledge update',
    'knowledge share',
    'knowledge confirmation',
    'argumentation',
    'resource use',
    'resource share',
    'resource aid',
    'resource request',
    'enact narration',
    'deterministic action'
]

### Evaluation

In [ ]:
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

def confusion_matrix_table(true_categories, predicted_categories, trace=None):
  for i in range(len(true_categories)):
    return confusion_matrix(true_categories, predicted_categories)

def cohens_kappa(true_categories, predicted_categories, trace=None):
  for i in range(len(true_categories)):
    return cohen_kappa_score(true_categories, predicted_categories)

def f1(true_categories, predicted_categories, trace=None):
  for i in range(len(true_categories)):
    return f1_score(true_categories, predicted_categories, average='weighted')

def precision(true_categories, predicted_categories, trace=None):
  for i in range(len(true_categories)):
    return precision_score(true_categories, predicted_categories, labels=label_list, average=None)

def recall(true_categories, predicted_categories, trace=None):
  for i in range(len(true_categories)):
    return recall_score(true_categories, predicted_categories, labels = label_list, average=None)

In [ ]:
recall_list = []
precision_list = []

for i in range(len(true_categories)):
  precision_list.append(precision(true_categories, predicted_categories))

for i in range(len(true_categories)):
  recall_list.append(recall(true_categories, predicted_categories))

In [ ]:
precision_list

In [ ]:
recall_list

In [ ]:
f1(true_categories, predicted_categories)

In [ ]:
semanticF1 = dspy.evaluate.SemanticF1(threshold=0.66, decompositional=False)

In [ ]:
evaluate = dspy.Evaluate(devset=trainset[0:50], metric = semanticF1, num_threads=1, provide_traceback=True)

In [ ]:
trainset[0].response

In [ ]:
evaluate(classify)

In [ ]:
optimizer_copro = dspy.COPRO(metric=semanticF1, breadth=10, depth=3, init_temperature=1.4, track_stats=True)
optimizer_fewshot = dspy.BootstrapFewShot(metric=semanticF1, metric_threshold=0.66, teacher_settings=None, max_bootstrapped_demos=16, max_labeled_demos=16, max_rounds=1)
optimizer_mipro = dspy.MIPROv2(metric=semanticF1, auto='light')

In [ ]:
!pip install dspy[optuna]

In [ ]:
import litellm

In [ ]:
litellm.drop_params=True

In [ ]:
optimized_classifier = optimizer_fewshot.compile(classify, trainset=trainset[51:101])

In [ ]:
evaluate(optimized_classifier)

In [ ]:
optimized_classifier.save("optimized_classifier.json")